### VAD — Voice Activity Detection

* Auto-detect when the user stops speaking.
* Uses silero-vad (neural net, runs on CPU).

Dynamically stop when user paused.

In [4]:
import numpy as np
import soundfile as sf
import torch
from pathlib import Path
from silero_vad import load_silero_vad, read_audio, get_speech_timestamps

 Load silero-vad model and test it

In [2]:
# Load the VAD model
vad_model = load_silero_vad()

c:\Users\USER\rag_course\rag_env\Lib\site-packages\torch\jit\_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


VAD detects speech in the sample audio

In [6]:
# Path to the audio file
audio_path = r'C:\Users\USER\rag_course\12b_voice_agents\data\mic_live.wav'

# Read with soundfile
audio_np, sr = sf.read(audio_path, dtype='float32')

print(f'Sample rate: {sr} Hz')
print(f'Duration:    {len(audio_np) / sr:.2f}s')
print(f'Shape:       {audio_np.shape}')
print()

# Convert to a torch tensor — what silero-vad expects
audio_tensor = torch.from_numpy(audio_np)

# Detect speech
timestamps = get_speech_timestamps(
    audio_tensor, 
    vad_model,
    sampling_rate=sr,
    return_seconds=True     # give times in seconds
)

print(f'Speech segments detected: {len(timestamps)}')
print()

for i, ts in enumerate(timestamps):
    print(f'  Segment {i+1}: {ts["start"]:.2f}s → {ts["end"]:.2f}s')

Sample rate: 16000 Hz
Duration:    5.00s
Shape:       (80000,)

Speech segments detected: 1

  Segment 1: 0.70s → 4.70s


In [7]:
import sounddevice as sd
import soundfile as sf
import torch
import numpy as np
import time

def record_until_silence(
    max_seconds: int = 15,
    silence_seconds: float = 1.5,
    min_speech_seconds: float = 0.5,
    device: int = 1,
) -> str:
    '''
    Record from mic until VAD detects silence, then stop.
    Returns the path to the saved WAV file.
    '''
    sample_rate = 16000
    chunk_samples = 512  # 32 ms per chunk at 16 kHz

    print(f'🎤 Recording (stop after {silence_seconds}s of silence)...')

    # Buffers
    chunks = []
    silence_duration = 0.0
    speech_duration = 0.0
    start_time = time.time()

    # Start streaming mic input
    with sd.InputStream(
        samplerate=sample_rate,
        channels=1,
        dtype='int16',
        device=device,
        blocksize=chunk_samples,
    ) as stream:
        while True:
            # Read one chunk (32 ms)
            chunk, _ = stream.read(chunk_samples)
            chunks.append(chunk)

            # Convert to float tensor for silero-vad
            audio_chunk = chunk.flatten().astype(np.float32) / 32768.0
            tensor = torch.from_numpy(audio_chunk)

            # Ask VAD: is this speech?
            prob = vad_model(tensor, sample_rate).item()
            is_speech = prob > 0.5

            if is_speech:
                speech_duration += chunk_samples / sample_rate
                silence_duration = 0.0
            else:
                silence_duration += chunk_samples / sample_rate

            # Stop conditions
            elapsed = time.time() - start_time
            if (speech_duration >= min_speech_seconds
                and silence_duration >= silence_seconds):
                print(f'✅ Detected end of speech ({elapsed:.2f}s total)')
                break
            if elapsed >= max_seconds:
                print(f'⏱️  Max duration reached ({max_seconds}s)')
                break

    # Save
    audio = np.concatenate(chunks, axis=0)
    out_path = r'C:\Users\USER\rag_course\12b_voice_agents\data\mic_vad.wav'
    sf.write(out_path, audio, sample_rate)

    duration = len(audio) / sample_rate
    print(f'💾 Saved: {out_path}')
    print(f'   Speech: {speech_duration:.2f}s  |  Total: {duration:.2f}s')

    return out_path


print('✅ record_until_silence ready')

✅ record_until_silence ready


In [8]:
# Record until you stop speaking
audio_path = record_until_silence()

🎤 Recording (stop after 1.5s of silence)...
✅ Detected end of speech (4.99s total)
💾 Saved: C:\Users\USER\rag_course\12b_voice_agents\data\mic_vad.wav
   Speech: 1.86s  |  Total: 4.00s


In [9]:
from IPython.display import Audio, display
display(Audio(audio_path))

Transcribe